In [5]:
import pandas as pd
import geopandas as gpd

In [6]:
oak_parcels = gpd.read_parquet("../data/processed/oak_parcels_sb79tiers_zones.parquet").copy()

#import zoning and SB79 description
#the zoning data was compiled into a csv with the help of ChatGPT and then manually verified
oak_zoning = pd.read_csv("../data/raw/cyplan101_zoning_oak.csv")
sb79_df = pd.read_csv("../data/raw/cyplan101_sb79.csv")

In [7]:
#merge use SB79 information onto parcel data
oak_parcels = oak_parcels.merge(
    sb79_df,
    left_on="tier_zone_label",
    right_on="tier_zone_label",
    how="left",
    validate="many_to_one"
)

#merge zoning information onto parcel data
oak_parcels = oak_parcels.merge(
    oak_zoning[['input_label', 'zoning_description', 'type', 'group_scale', 'residential_permission_notes', 'max_height_ft_clean']],
    left_on="basezone",
    right_on="input_label",
    how="left",
    validate="many_to_one"
)

In [8]:
#ensure the scale is numeric
oak_parcels['sb79_scale'] = pd.to_numeric(oak_parcels['sb79_scale'], errors='coerce')

oak_parcels['group_scale'] = pd.to_numeric(oak_parcels['group_scale'], errors='coerce')

In [10]:
#if the zone type is not eligible for upzoning, set the SB79 scale to None
mask = ~oak_parcels['type'].isin(['commercial', 'mixed_use', 'residential'])
oak_parcels.loc[mask, ['sb79_scale', 'max_height_sb79']] = None
oak_parcels.head()

,APN,APN_SORT,parcel_centroid_x,parcel_centroid_y,Land,Imps,UseCode,SitusAddress,geometry,OBJECTID,...,tier,radius,sb79_scale,max_height_sb79,input_label,zoning_description,type,group_scale,residential_permission_notes,max_height_ft_clean
0,1-1111-1,001 111100100,-1.361081e+07,4.550375e+06,161934,438407,3900,351 OAK ST OAKLAND 94607,"POLYGON ((-122.2683 37.7945, -122.26737 37.794...",149,...,1,1/4,5.0,75.0,C-45,Community Shopping,commercial,NaN,"Requested label is unverified, repealed, or ha...",NaN
1,1-1111-10,001 111101000,-1.361081e+07,4.550375e+06,156615,365435,7301,311 OAK ST 105 OAKLAND 94607,"POLYGON ((-122.2683 37.7945, -122.26737 37.794...",149,...,1,1/4,5.0,75.0,C-45,Community Shopping,commercial,NaN,"Requested label is unverified, repealed, or ha...",NaN
2,1-1111-11,001 111101100,-1.361081e+07,4.550375e+06,76443,178367,7301,311 OAK ST 106 OAKLAND 94607,"POLYGON ((-122.2683 37.7945, -122.26737 37.794...",149,...,1,1/4,5.0,75.0,C-45,Community Shopping,commercial,NaN,"Requested label is unverified, repealed, or ha...",NaN
3,1-1111-12,001 111101200,-1.361081e+07,4.550375e+06,83336,194452,7301,311 OAK ST 107 OAKLAND 94607,"POLYGON ((-122.2683 37.7945, -122.26737 37.794...",149,...,1,1/4,5.0,75.0,C-45,Community Shopping,commercial,NaN,"Requested label is unverified, repealed, or ha...",NaN
4,1-1111-13,001 111101300,-1.361081e+07,4.550375e+06,71101,165902,7301,311 OAK ST 108 OAKLAND 94607,"POLYGON ((-122.2683 37.7945, -122.26737 37.794...",149,...,1,1/4,5.0,75.0,C-45,Community Shopping,commercial,NaN,"Requested label is unverified, repealed, or ha...",NaN


In [ ]:
#save cleaned and joined gdf as a parqet file to use for analysis
oak_parcels.to_parquet("../data/processed/oak_parcels.parquet")